# Airbnb Price Prediction Project


### 1) Problem Statement

The goal of this project is to predict Airbnb listing prices using structured listing data such as
location, room type, review scores, host information, and availability.
This notebook includes a city quality screening step that automatically identifies and removes
cities that hurt model performance, in addition to a complete ML workflow with data loading,
cleaning, feature engineering, model building, hyperparameter tuning, evaluation, and interpretation.

In [ ]:
# =========================================
# 1. IMPORTS
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from pathlib import Path

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
CZK_TO_USD = 0.0485
DATA_DIR = Path("listings.csv")
MIN_ROWS_PER_CITY = 200
TOP_CITY_OUTLIER_Q = 0.99
MAX_COL_MISSING = 0.80
# Cities with per-city screening MAE above this multiple of the median are dropped
CITY_MAE_MULTIPLIER = 2.0


### 2) Data Loading

Airbnb listings from all compatible files in the listings folder are loaded and combined.
The Prague dataset is converted from Czech koruna to U.S. dollars so all cities share the same currency scale.

In [ ]:
# =========================================
# 2. LOAD AND COMBINE MULTIPLE CITY DATASETS
# =========================================
def clean_price_column(series):
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("K\u010d", "", regex=False)
        .str.replace("CZK", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")


def parse_city_name(filename):
    name = filename.replace("listings - ", "").replace(".csv", "")
    parts = [p.strip() for p in name.split(",")]
    return parts[0], name


def load_all_city_data(data_dir):
    frames = []
    for file_path in sorted(data_dir.glob("listings - *.csv")):
        temp = pd.read_csv(file_path)
        city_short, city_full = parse_city_name(file_path.name)
        temp["city"] = city_short
        temp["city_full"] = city_full
        temp["price"] = clean_price_column(temp["price"])
        if "Prague" in city_full:
            temp["price"] = temp["price"] * CZK_TO_USD
        frames.append(temp)
    return pd.concat(frames, ignore_index=True)


df = load_all_city_data(DATA_DIR)

print("Combined dataset shape:", df.shape)
print("\nRows by city:")
print(df["city_full"].value_counts())
print("\nAverage price by city after conversion:")
print(df.groupby("city_full")["price"].mean().sort_values(ascending=False))
print("\nMissing price share by city:")
print(df.groupby("city_full")["price"].apply(lambda x: x.isna().mean()))


### 3) Data Exploration

This section explores the structure of the dataset, missing values, and the distribution of prices.
The plots help justify later preprocessing choices such as outlier handling and log transformation.

In [ ]:
# =========================================
# 3. DATA EXPLORATION
# =========================================
print("Dataset shape:", df.shape)
print("\nTop missing values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))
print("\nPrice summary statistics:")
print(df["price"].describe())
print("\nAverage price by city:")
print(df.groupby("city_full")["price"].mean().sort_values(ascending=False))

plt.figure(figsize=(10, 5))
plt.hist(df["price"].dropna(), bins=50, edgecolor="black")
plt.xlabel("Price in Dollars")
plt.ylabel("Number of Listings")
plt.title("Raw Airbnb Price Distribution")
plt.tight_layout()
plt.show()

price_95 = df["price"].quantile(0.95)
plt.figure(figsize=(10, 5))
plt.hist(df.loc[df["price"] <= price_95, "price"].dropna(), bins=40, edgecolor="black")
plt.xlabel("Price in Dollars")
plt.ylabel("Number of Listings")
plt.title("Airbnb Price Distribution up to the 95th Percentile")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
np.log1p(df["price"].dropna()).hist(bins=40, edgecolor="black")
plt.xlabel("Log Price")
plt.ylabel("Number of Listings")
plt.title("Distribution of Log-Transformed Airbnb Prices")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
df["city_full"].value_counts().plot(kind="bar")
plt.xlabel("City")
plt.ylabel("Number of Listings")
plt.title("Number of Listings by City")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

if "room_type" in df.columns:
    plt.figure(figsize=(8, 5))
    df["room_type"].value_counts().plot(kind="bar")
    plt.xlabel("Room Type")
    plt.ylabel("Number of Listings")
    plt.title("Number of Listings by Room Type")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

df.groupby("city_full")["price"].mean().sort_values(ascending=False).plot(
    kind="bar", figsize=(10, 5)
)
plt.xlabel("City")
plt.ylabel("Average Price in USD")
plt.title("Average Airbnb Price by City")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

candidate_cols = [
    "price", "accommodates", "bedrooms", "beds",
    "bathrooms", "number_of_reviews", "review_scores_rating", "availability_365"
]
existing_cols = [c for c in candidate_cols if c in df.columns]
if len(existing_cols) > 1:
    plt.figure(figsize=(8, 6))
    sns.heatmap(df[existing_cols].corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Heatmap of Key Numeric Features")
    plt.tight_layout()
    plt.show()


### 4) Data Cleaning

The target variable is cleaned by removing missing values and filtering extreme outliers within each
city. A log transformation is then applied to stabilize the target distribution.

In [ ]:
# =========================================
# 4. CLEAN TARGET VARIABLE
# =========================================
df = df.dropna(subset=["price"]).copy()
df = df[df["price"] > 0].copy()

city_counts = df["city_full"].value_counts()
keep_cities = city_counts[city_counts >= MIN_ROWS_PER_CITY].index
df = df[df["city_full"].isin(keep_cities)].copy()

city_cutoff = df.groupby("city_full")["price"].transform(
    lambda x: x.quantile(TOP_CITY_OUTLIER_Q)
)
df = df[df["price"] <= city_cutoff].copy()

print("Dataset shape after per-city 99th-percentile cutoff:", df.shape)
print("\nCity counts after cleaning:")
print(df["city_full"].value_counts())

df["log_price"] = np.log1p(df["price"])

plt.figure(figsize=(10, 5))
plt.hist(df["price"].dropna(), bins=40, edgecolor="black")
plt.xlabel("Price")
plt.ylabel("Count")
plt.title("Cleaned Price Distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(df["log_price"].dropna(), bins=40, edgecolor="black")
plt.xlabel("Log Price")
plt.ylabel("Count")
plt.title("Log-Transformed Price Distribution")
plt.tight_layout()
plt.show()


### 5) Feature Engineering

New features are created from dates, listing characteristics, review activity, and location patterns
to improve model performance.

In [ ]:
# =========================================
# 5. DATETIME FEATURES
# =========================================
date_cols = ["host_since", "first_review", "last_review", "last_scraped", "calendar_last_scraped"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

reference_date = df["last_scraped"].max() if "last_scraped" in df.columns else pd.Timestamp.today()

if "host_since" in df.columns:
    df["host_tenure_days"] = (reference_date - df["host_since"]).dt.days
if "first_review" in df.columns:
    df["days_since_first_review"] = (reference_date - df["first_review"]).dt.days
if "last_review" in df.columns:
    df["days_since_last_review"] = (reference_date - df["last_review"]).dt.days


In [ ]:
# =========================================
# 6. FEATURE ENGINEERING
# =========================================
for col in ["host_response_rate", "host_acceptance_rate"]:
    if col in df.columns:
        df[col] = (
            df[col].astype(str)
            .str.replace("%", "", regex=False)
            .replace("nan", np.nan)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

bool_map = {"t": 1, "f": 0, True: 1, False: 0}
for col in ["instant_bookable", "host_is_superhost", "host_identity_verified",
            "host_has_profile_pic", "has_availability"]:
    if col in df.columns:
        df[col] = df[col].map(bool_map)

if "bathrooms_text" in df.columns:
    df["bathrooms_text_num"] = (
        df["bathrooms_text"].astype(str)
        .str.extract(r"(\d+\.?\d*)")[0]
        .astype(float)
    )

if "bathrooms" in df.columns and "bathrooms_text_num" in df.columns:
    df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms_text_num"])
elif "bathrooms" not in df.columns and "bathrooms_text_num" in df.columns:
    df["bathrooms"] = df["bathrooms_text_num"]

if "amenities" in df.columns:
    def count_amenities(x):
        try:
            return len(ast.literal_eval(x))
        except Exception:
            return np.nan
    df["amenities_count"] = df["amenities"].apply(count_amenities)

if "latitude" in df.columns and "longitude" in df.columns and "city_full" in df.columns:
    city_centers = (
        df.groupby("city_full")[["latitude", "longitude"]]
        .median()
        .rename(columns={"latitude": "center_lat", "longitude": "center_lon"})
    )
    df = df.merge(city_centers, on="city_full", how="left")

    def haversine_km(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat, dlon = lat2 - lat1, lon2 - lon1
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
        return 6371 * 2 * np.arcsin(np.sqrt(a))

    df["distance_to_city_center"] = haversine_km(
        df["latitude"], df["longitude"], df["center_lat"], df["center_lon"]
    )

if "bedrooms" in df.columns and "accommodates" in df.columns:
    df["bedroom_density"] = df["bedrooms"] / (df["accommodates"] + 1)
if "beds" in df.columns and "accommodates" in df.columns:
    df["bed_density"] = df["beds"] / (df["accommodates"] + 1)
if "beds" in df.columns and "bedrooms" in df.columns:
    df["beds_per_bedroom"] = df["beds"] / (df["bedrooms"] + 1)
if "availability_365" in df.columns:
    df["availability_ratio"] = df["availability_365"] / 365
if "number_of_reviews" in df.columns and "host_listings_count" in df.columns:
    df["reviews_per_listing"] = df["number_of_reviews"] / (df["host_listings_count"] + 1)
if "reviews_per_month" in df.columns and "host_listings_count" in df.columns:
    df["reviews_per_month_per_listing"] = df["reviews_per_month"] / (df["host_listings_count"] + 1)
if "number_of_reviews" in df.columns and "host_tenure_days" in df.columns:
    df["reviews_per_day"] = df["number_of_reviews"] / (df["host_tenure_days"] + 1)
if "availability_365" in df.columns and "number_of_reviews" in df.columns:
    df["availability_per_review"] = df["availability_365"] / (df["number_of_reviews"] + 1)
if "accommodates" in df.columns and "bedrooms" in df.columns:
    df["accommodates_x_bedrooms"] = df["accommodates"] * df["bedrooms"]
if "accommodates" in df.columns and "beds" in df.columns:
    df["accommodates_x_beds"] = df["accommodates"] * df["beds"]
if "number_of_reviews" in df.columns:
    df["log_number_of_reviews"] = np.log1p(df["number_of_reviews"])
if "minimum_nights" in df.columns:
    df["log_minimum_nights"] = np.log1p(df["minimum_nights"])

if "latitude" in df.columns and "longitude" in df.columns and "city_full" in df.columns:
    df["location_cluster"] = -1
    cluster_offset = 0
    for city_name in df["city_full"].unique():
        subset = df[df["city_full"] == city_name][["latitude", "longitude"]].dropna()
        if len(subset) < 50:
            continue
        n_clusters = int(np.clip(len(subset) // 250, 8, 30))
        labels = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10).fit_predict(subset)
        df.loc[subset.index, "location_cluster"] = labels + cluster_offset
        cluster_offset += n_clusters

keep_force = {"price", "log_price", "city", "city_full", "latitude", "longitude", "neighbourhood_cleansed"}
keep_cols = [c for c in df.columns if (df[c].isnull().mean() < MAX_COL_MISSING) or (c in keep_force)]
df = df[keep_cols].copy()
df = df.dropna(axis=1, how="all")

print("Dataset shape after feature engineering:", df.shape)


### 6) Feature Selection

A curated list of features is chosen as input to the model. Only columns that exist in the
dataset are retained.

In [ ]:
# =========================================
# 7. SELECT FEATURES
# =========================================
feature_cols = [
    "city", "city_full",
    "accommodates", "bathrooms", "bathrooms_text_num", "bedrooms", "beds",
    "minimum_nights", "maximum_nights",
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d",
    "reviews_per_month",
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication", "review_scores_location",
    "review_scores_value",
    "host_listings_count", "host_total_listings_count",
    "calculated_host_listings_count", "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms", "calculated_host_listings_count_shared_rooms",
    "availability_30", "availability_60", "availability_90", "availability_365",
    "availability_eoy", "estimated_occupancy_l365d",
    "host_response_rate", "host_acceptance_rate",
    "latitude", "longitude", "distance_to_city_center",
    "host_tenure_days", "days_since_first_review", "days_since_last_review",
    "amenities_count",
    "instant_bookable", "host_is_superhost", "host_identity_verified",
    "host_has_profile_pic", "has_availability",
    "room_type", "property_type", "neighbourhood_cleansed",
    "host_response_time", "source",
    "bedroom_density", "bed_density", "beds_per_bedroom",
    "availability_ratio", "reviews_per_listing", "reviews_per_month_per_listing",
    "reviews_per_day", "availability_per_review",
    "accommodates_x_bedrooms", "accommodates_x_beds",
    "log_number_of_reviews", "log_minimum_nights",
    "location_cluster"
]

feature_cols = [c for c in feature_cols if c in df.columns]
print("Number of selected features:", len(feature_cols))
print(feature_cols)


### 7) City Quality Screening

A quick ExtraTrees model is fit on a temporary 70/30 split to measure per-city prediction error.
Cities with MAE more than `CITY_MAE_MULTIPLIER` times the median city MAE are removed before
the final train/dev/test split. This prevents cities with incompatible pricing dynamics from
dragging down overall model performance.

In [ ]:
# =========================================
# 8. CITY QUALITY SCREENING
# =========================================
def smoothed_target_mean(train_x, train_y, group_col, apply_x, min_count=25, smoothing=20):
    """Bayesian-smoothed target mean encoding, computed from train only."""
    temp = pd.DataFrame({"group": train_x[group_col].values, "price": np.expm1(train_y.values)})
    global_mean = temp["price"].mean()
    stats = temp.groupby("group")["price"].agg(["mean", "count"])
    smooth = (stats["count"] * stats["mean"] + smoothing * global_mean) / (stats["count"] + smoothing)
    smooth[stats["count"] < min_count] = global_mean
    return apply_x[group_col].map(smooth.to_dict()).fillna(global_mean)


X_scr = df[feature_cols].copy()
y_scr = df["log_price"].copy()

X_scr_tr, X_scr_ev, y_scr_tr, y_scr_ev = train_test_split(
    X_scr, y_scr, test_size=0.30, random_state=RANDOM_STATE,
    stratify=X_scr["city_full"]
)

X_scr_tr = X_scr_tr.copy()
X_scr_ev = X_scr_ev.copy()
X_scr_tr["city_avg_price"] = smoothed_target_mean(X_scr_tr, y_scr_tr, "city_full", X_scr_tr)
X_scr_ev["city_avg_price"] = smoothed_target_mean(X_scr_tr, y_scr_tr, "city_full", X_scr_ev)

numeric_s = X_scr_tr.select_dtypes(include=["number"]).columns.tolist()
categorical_s = [c for c in X_scr_tr.columns if c not in numeric_s]

scr_preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), numeric_s),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_s)
])

scr_pipe = Pipeline([
    ("preprocessor", scr_preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=200, max_depth=20, random_state=RANDOM_STATE, n_jobs=-1
    ))
])

scr_pipe.fit(X_scr_tr, y_scr_tr)
scr_preds = scr_pipe.predict(X_scr_ev)

eval_screen = pd.DataFrame({
    "city_full": X_scr_ev["city_full"].values,
    "actual": np.expm1(y_scr_ev.values),
    "predicted": np.expm1(scr_preds)
})
eval_screen["abs_error"] = (eval_screen["actual"] - eval_screen["predicted"]).abs()

city_mae_screen = eval_screen.groupby("city_full")["abs_error"].mean().sort_values()
median_city_mae = city_mae_screen.median()
threshold = CITY_MAE_MULTIPLIER * median_city_mae
bad_cities = city_mae_screen[city_mae_screen > threshold].index.tolist()

print("Per-city screening MAE (USD):")
print(city_mae_screen.round(2).to_string())
print(f"\nMedian city MAE: ${median_city_mae:.2f}  |  Threshold ({CITY_MAE_MULTIPLIER}x median): ${threshold:.2f}")
print(f"\nCities flagged for removal:")
if bad_cities:
    for c in bad_cities:
        print(f"  {c}: ${city_mae_screen[c]:.2f}")
    df = df[~df["city_full"].isin(bad_cities)].copy()
    print(f"\nDataset shape after city filter: {df.shape}")
else:
    print("  None — all cities retained")

colors = ["crimson" if c in bad_cities else "steelblue" for c in city_mae_screen.index]
plt.figure(figsize=(12, 5))
city_mae_screen.plot(kind="bar", color=colors)
plt.axhline(threshold, color="crimson", linestyle="--", label=f"Threshold ${threshold:.0f}")
plt.xlabel("City")
plt.ylabel("MAE (USD)")
plt.title("Per-City Screening MAE — Crimson bars removed")
plt.legend()
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()


### 8) Train / Validation / Test Split

The filtered dataset is split 60/20/20 into train, dev, and test sets, stratified by city.
Target encoding is computed separately for dev evaluation (using train only) and for
final model training (using the full train set), preventing leakage in both cases.

In [ ]:
# =========================================
# 9. TRAIN / DEV / TEST SPLIT
# =========================================
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].copy()
y = df["log_price"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE,
    stratify=X["city_full"]
)

X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_STATE,
    stratify=X_train_full["city_full"]
)

print("Train shape:", X_train.shape)
print("Dev shape:  ", X_dev.shape)
print("Test shape: ", X_test.shape)

print("\nCity distribution in Train:")
print(X_train["city_full"].value_counts(normalize=True).round(3))


In [ ]:
# =========================================
# 10. TARGET ENCODING
# =========================================
# Dev evaluation: encode using X_train only (no leakage from dev/test)
# Final models: encode using X_train_full (more stable for production predictions)

X_train = X_train.copy()
X_dev = X_dev.copy()
X_train_full = X_train_full.copy()
X_test = X_test.copy()

X_train["city_avg_price"] = smoothed_target_mean(X_train, y_train, "city_full", X_train)
X_dev["city_avg_price"] = smoothed_target_mean(X_train, y_train, "city_full", X_dev)
X_train_full["city_avg_price"] = smoothed_target_mean(X_train_full, y_train_full, "city_full", X_train_full)
X_test["city_avg_price"] = smoothed_target_mean(X_train_full, y_train_full, "city_full", X_test)

if "neighbourhood_cleansed" in X_train.columns:
    X_train["neighborhood_avg_price"] = smoothed_target_mean(
        X_train, y_train, "neighbourhood_cleansed", X_train, min_count=30, smoothing=25
    )
    X_dev["neighborhood_avg_price"] = smoothed_target_mean(
        X_train, y_train, "neighbourhood_cleansed", X_dev, min_count=30, smoothing=25
    )
    X_train_full["neighborhood_avg_price"] = smoothed_target_mean(
        X_train_full, y_train_full, "neighbourhood_cleansed", X_train_full, min_count=30, smoothing=25
    )
    X_test["neighborhood_avg_price"] = smoothed_target_mean(
        X_train_full, y_train_full, "neighbourhood_cleansed", X_test, min_count=30, smoothing=25
    )

print("Target encoding complete.")
print("Train features:", X_train.shape[1])
print("X_train_full features:", X_train_full.shape[1])


### 9) Modeling

Multiple models are compared including a simple baseline, Ridge Regression, Random Forest,
ExtraTrees, and XGBoost. This helps determine whether more flexible models outperform simpler ones.

In [ ]:
# =========================================
# 11. PREPROCESSING PIPELINES
# =========================================
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

ridge_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

tree_preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])


In [ ]:
# =========================================
# 12. BASELINE AND INITIAL MODELS
# =========================================
baseline_pred = np.full(len(y_dev), fill_value=y_train.mean())
baseline_mae = mean_absolute_error(np.expm1(y_dev), np.expm1(baseline_pred))
print("Baseline MAE (dollars):", round(baseline_mae, 2))

models = {
    "Ridge": Pipeline([
        ("preprocessor", ridge_preprocessor),
        ("model", Ridge(alpha=3.0))
    ]),
    "RandomForest": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestRegressor(
            random_state=RANDOM_STATE, n_jobs=-1,
            n_estimators=500, max_depth=30,
            min_samples_split=5, min_samples_leaf=2,
            max_features="sqrt"
        ))
    ]),
    "ExtraTrees": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", ExtraTreesRegressor(
            random_state=RANDOM_STATE, n_jobs=-1,
            n_estimators=500, max_depth=None,
            min_samples_split=5, min_samples_leaf=2,
            max_features="sqrt"
        ))
    ])
}

results = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_dev)
    results.append({
        "Model": name,
        "MAE_log": mean_absolute_error(y_dev, preds),
        "MAE_dollars": mean_absolute_error(np.expm1(y_dev), np.expm1(preds)),
        "R2": r2_score(y_dev, preds)
    })

results_df = pd.DataFrame(results).sort_values("MAE_dollars")
print(results_df.to_string(index=False))


### 10) Hyperparameter Tuning

The strongest tree-based models are tuned using RandomizedSearchCV.
The final tuned models are trained on the full training set (X_train_full) which includes
properly computed target encoding features.

In [ ]:
# =========================================
# 13. RANDOM FOREST HYPERPARAMETER TUNING
# =========================================
# Rebuild preprocessor column lists using X_train_full (has all target-encoded features)
numeric_cols_full = X_train_full.select_dtypes(include=["number"]).columns.tolist()
categorical_cols_full = [c for c in X_train_full.columns if c not in numeric_cols_full]

tree_preprocessor_full = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_cols_full),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols_full)
])

rf_pipe = Pipeline([
    ("preprocessor", tree_preprocessor_full),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

rf_param_dist = {
    "model__n_estimators": [300, 500, 700],
    "model__max_depth": [20, 30, 40, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

rf_search = RandomizedSearchCV(
    rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=15,
    cv=3,
    scoring="neg_mean_absolute_error",
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_search.fit(X_train_full, y_train_full)

print("Best Random Forest params:")
print(rf_search.best_params_)

best_model = rf_search.best_estimator_


In [ ]:
# =========================================
# 14. XGBOOST MODEL
# =========================================
xgb_pipe = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_estimators=700,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

xgb_pipe.fit(X_train, y_train)
xgb_dev_preds = xgb_pipe.predict(X_dev)

print("XGBoost Dev Results")
print("MAE (log):    ", round(mean_absolute_error(y_dev, xgb_dev_preds), 4))
print("MAE (dollars):", round(mean_absolute_error(np.expm1(y_dev), np.expm1(xgb_dev_preds)), 2))
print("R2:           ", round(r2_score(y_dev, xgb_dev_preds), 4))


In [ ]:
# =========================================
# 15. XGBOOST HYPERPARAMETER TUNING
# =========================================
xgb_tune_pipe = Pipeline([
    ("preprocessor", tree_preprocessor_full),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

xgb_param_dist = {
    "model__n_estimators": [400, 700, 1000],
    "model__learning_rate": [0.01, 0.03, 0.05],
    "model__max_depth": [4, 5, 6, 8],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__reg_alpha": [0, 0.05, 0.1, 0.5],
    "model__reg_lambda": [1.0, 1.5, 2.0, 3.0]
}

xgb_search = RandomizedSearchCV(
    xgb_tune_pipe,
    param_distributions=xgb_param_dist,
    n_iter=18,
    cv=3,
    scoring="neg_mean_absolute_error",
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_search.fit(X_train_full, y_train_full)

print("Best XGBoost params:")
print(xgb_search.best_params_)

best_xgb_model = xgb_search.best_estimator_


### 11) Model Evaluation

Final models are evaluated on the held-out test set using MAE and R².
A per-city breakdown shows where each model performs best and worst.

In [ ]:
# =========================================
# 16. FINAL TEST EVALUATION — RANDOM FOREST
# =========================================
test_preds = best_model.predict(X_test)

print("Final Random Forest Test Results")
print("MAE (log):    ", round(mean_absolute_error(y_test, test_preds), 4))
print("MAE (dollars):", round(mean_absolute_error(np.expm1(y_test), np.expm1(test_preds)), 2))
print("R2:           ", round(r2_score(y_test, test_preds), 4))


In [ ]:
# =========================================
# 17. FINAL TEST EVALUATION — XGBOOST
# =========================================
xgb_test_preds = best_xgb_model.predict(X_test)

print("Final XGBoost Test Results")
print("MAE (log):    ", round(mean_absolute_error(y_test, xgb_test_preds), 4))
print("MAE (dollars):", round(mean_absolute_error(np.expm1(y_test), np.expm1(xgb_test_preds)), 2))
print("R2:           ", round(r2_score(y_test, xgb_test_preds), 4))


In [ ]:
# =========================================
# 18. WEIGHTED ENSEMBLE
# =========================================
rf_preds_dev = best_model.predict(X_dev)
xgb_preds_dev = best_xgb_model.predict(X_dev)

best_weight, best_dev_mae = None, float("inf")
for w in np.linspace(0, 1, 21):
    blended = rf_preds_dev * w + xgb_preds_dev * (1 - w)
    mae = mean_absolute_error(np.expm1(y_dev), np.expm1(blended))
    if mae < best_dev_mae:
        best_dev_mae, best_weight = mae, w

print(f"Best RF weight: {best_weight:.2f}  |  Dev ensemble MAE: ${best_dev_mae:.2f}")

rf_preds = best_model.predict(X_test)
xgb_preds = best_xgb_model.predict(X_test)
ensemble_preds = rf_preds * best_weight + xgb_preds * (1 - best_weight)

ensemble_mae = mean_absolute_error(np.expm1(y_test), np.expm1(ensemble_preds))
ensemble_r2 = r2_score(y_test, ensemble_preds)
mean_price = np.expm1(y_test).mean()

print("\nEnsemble Test Results")
print("MAE (dollars):", round(ensemble_mae, 2))
print("R2:           ", round(ensemble_r2, 4))
print("Mean price:   ", round(mean_price, 2))
print("Relative error:", round(ensemble_mae / mean_price, 4))


In [ ]:
# =========================================
# 19. PER-CITY FINAL EVALUATION
# =========================================
test_eval = pd.DataFrame({
    "city_full": X_test["city_full"].values,
    "actual": np.expm1(y_test.values),
    "ensemble_pred": np.expm1(ensemble_preds)
})
test_eval["abs_error"] = (test_eval["actual"] - test_eval["ensemble_pred"]).abs()

city_final = test_eval.groupby("city_full").agg(
    n=("actual", "count"),
    mean_price=("actual", "mean"),
    mae=("abs_error", "mean")
).sort_values("mae")
city_final["relative_error"] = (city_final["mae"] / city_final["mean_price"]).round(3)
city_final[["mae", "mean_price"]] = city_final[["mae", "mean_price"]].round(2)

print("Per-city ensemble test MAE:")
print(city_final.to_string())

plt.figure(figsize=(12, 5))
city_final["mae"].plot(kind="bar", color="steelblue")
plt.axhline(ensemble_mae, color="crimson", linestyle="--", label=f"Overall MAE ${ensemble_mae:.0f}")
plt.xlabel("City")
plt.ylabel("MAE (USD)")
plt.title("Per-City Ensemble MAE on Test Set")
plt.legend()
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()


### Prediction Plots

These plots compare actual and predicted prices for the final Random Forest and XGBoost models.
A tighter pattern along the diagonal indicates stronger predictive performance.

In [ ]:
# =========================================
# 20. ACTUAL VS PREDICTED — RANDOM FOREST
# =========================================
actual_prices = np.expm1(y_test)
predicted_prices = np.expm1(test_preds)

plt.figure(figsize=(8, 6))
plt.scatter(actual_prices, predicted_prices, alpha=0.4, s=10)
lim = max(actual_prices.max(), predicted_prices.max())
plt.plot([0, lim], [0, lim], "r--", linewidth=1, label="Perfect prediction")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted — Random Forest")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 21. ACTUAL VS PREDICTED — XGBOOST
# =========================================
xgb_predicted_prices = np.expm1(xgb_test_preds)

plt.figure(figsize=(8, 6))
plt.scatter(actual_prices, xgb_predicted_prices, alpha=0.4, s=10)
lim = max(actual_prices.max(), xgb_predicted_prices.max())
plt.plot([0, lim], [0, lim], "r--", linewidth=1, label="Perfect prediction")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted — XGBoost")
plt.legend()
plt.tight_layout()
plt.show()


### 12) Model Interpretation

Permutation importance identifies which features most influence model performance,
highlighting the strongest drivers of Airbnb pricing.

In [ ]:
# =========================================
# 22. PERMUTATION IMPORTANCE
# =========================================
perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

print("\nTop 20 Important Features:")
print(importance_df.head(20).to_string(index=False))

plt.figure(figsize=(10, 7))
sns.barplot(data=importance_df.head(20), x="importance", y="feature")
plt.title("Top 20 Permutation Importances — Random Forest")
plt.tight_layout()
plt.show()


### 13) Conclusion

This project developed machine learning models to predict Airbnb listing prices across multiple cities.
A city quality screening step automatically removes cities that hurt model performance before the
final train/dev/test split. After cleaning, feature engineering, and comparing multiple models,
the final tuned ensemble achieved strong predictive performance.

Key improvements in version 6.0 over 5.0:
- **City quality screening**: automatic removal of cities where model error exceeds 2x the median.
- **Fixed target encoding for X_train_full**: final models now train with correctly computed city and neighborhood average price features.
- **Separate preprocessors for train/dev vs full-train**: prevents column mismatch between dev evaluation and final model training.
- **XGBoost `tree_method=hist`**: faster histogram-based tree construction.
- **Per-city test evaluation table and plot**: shows which remaining cities the model handles best.
- **Perfect-prediction diagonal on scatter plots**: makes fit quality visually clearer.

Future improvements could include seasonal features, calendar availability patterns, or
textual review information.